In [1]:
import pandas as pd
import torch

In [2]:
train_df = pd.read_csv("/content/train.csv")

Q1. Label Encoding

Convert the answer column in train.csv into numeric labels using the following mapping:

A = 0

B = 1

C = 2

D = 3

E = 4

What is the encoded numeric label for the row at index 150?

In [3]:
label_map = {"A": 0 , "B": 1, "C":2, "D":3, "E":4}

train_df["answer"] = train_df["answer"].apply(lambda answer: label_map[answer])

In [4]:
print(f"Encoded numeric label for row at index 150 is: {train_df.iloc[150]["answer"]}")

Encoded numeric label for row at index 150 is: 2


Q2. Prompt-Option Formatting

For row index 0, create the Option B input using exactly this format:
str(prompt) + " [SEP] " + str(option_B)

What is the exact character length of this formatted input string?
*

In [5]:
prompt_0 = train_df.iloc[0]["prompt"]
optionB_0 = train_df.iloc[0]["B"]

optionB = str(prompt_0) + " [SEP] " + str(optionB_0)

print(optionB)

print(f'Exact Character length of this formatted input string:{len(optionB)}')

Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options. [SEP] Martin Heidegger believes that humans do not exist inside time, but that they are time. The relationship to the past is a present awareness of having been, and the relationship to the future involves anticipating a potential possibility, task, or engagement.
Exact Character length of this formatted input string:407


Q3. Single-Row MCQ Tokenization

Using bert-base-uncased, tokenize the five formatted inputs for row index 0 with:

padding = "max_length"

truncation = True

max_length = 128

return_tensors = "pt"


After reshaping for a multiple-choice model, the final input_ids tensor has shape:
[1, 5, 128]

What is the value of the second dimension?

In [6]:
from transformers import AutoTokenizer
MODEL = "google-bert/bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL)

In [7]:
options = torch.randn(1,5,128)

for option in ["A", "B", "C", "D", "E"]:
  formatted_option = str(prompt_0) + " [SEP] " + str(train_df.iloc[0][option])

  tokenized_input = tokenizer(
      formatted_option,
      padding = "max_length",
      truncation = True,
      max_length = 128,
      return_tensors = "pt"
  )
  options[0,label_map[option]]= tokenized_input["input_ids"]

print(options.shape)

torch.Size([1, 5, 128])


Q4. Batch MCQ Tokenization

Tokenize the first 16 rows of train.csv as multiple-choice examples.

Each row has 5 choices.

Each choice is tokenized to length 128.

The final input_ids tensor has shape:
[16, 5, 128]

How many total token positions are in this tensor?

In [8]:
num = 16
input = []

for i in range(num):
  row = train_df.iloc[i]

  prompt = row["prompt"]
  options= []
  for item in label_map.keys():
    options.append(row[item])

  tokenized_input = tokenizer(
    options,
    padding = "max_length",
    truncation = True,
    max_length = 128,
    return_tensors = "pt"
  )

  input.append(tokenized_input["input_ids"])


input_tokens = torch.stack(input, dim =0)

input_tokens_shape = input_tokens.shape

print(f"Total token positions in this tensor: {input_tokens_shape[0]*input_tokens_shape[1]*input_tokens_shape[2]}")

Total token positions in this tensor: 10240


Q5. Multiple-Choice Logits

Load bert-base-uncased using AutoModelForMultipleChoice.

Tokenize row index 0 as 5 choices and pass it through the model.

The output logits tensor has shape:
[1, 5]

How many logits are produced for one question?

In [9]:
from transformers import AutoModelForMultipleChoice
model = AutoModelForMultipleChoice.from_pretrained(MODEL)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForMultipleChoice LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [10]:
options = []
for item in label_map.keys():
  option = train_df.iloc[0][item]
  format_option = prompt_0 + " [SEP] " + option

  options.append(format_option)

input_tokens = tokenizer(
    options,
    padding = "max_length",
    truncation = True,
    max_length = 128,
    return_tensors = "pt"
)
input = {}

input["input_ids"] = input_tokens["input_ids"].unsqueeze(0)
input["attention_mask"] = input_tokens["attention_mask"].unsqueeze(0)
input["token_type_ids"] = input_tokens["token_type_ids"].unsqueeze(0)

with torch.no_grad():
  output = model(**input)

print(output)

MultipleChoiceModelOutput(loss=None, logits=tensor([[-0.2282, -0.2169, -0.1989, -0.2136, -0.1829]]), hidden_states=None, attentions=None)


In [11]:
output.logits.shape

torch.Size([1, 5])

Q6. Supervised Loss Tensor

For row index 0, pass the tokenized 5-choice input into AutoModelForMultipleChoice along with the correct encoded label.

The model returns a scalar loss tensor.

How many dimensions does this loss tensor have?

In [12]:
answer_label = torch.tensor(train_df.iloc[0]["answer"]).reshape(-1)

with torch.no_grad():
  output = model(**input, labels = answer_label)

print(output)

print(f"Dimensions of the loss tensor is: {output.loss.shape}")

MultipleChoiceModelOutput(loss=tensor(1.6183), logits=tensor([[-0.2282, -0.2169, -0.1989, -0.2136, -0.1829]]), hidden_states=None, attentions=None)
Dimensions of the loss tensor is: torch.Size([])


Q7. LoRA Trainable Parameters

Apply LoRA to the bert-base-uncased multiple-choice model using:

r = 8

lora_alpha = 16

target_modules = ["query", "value"]

lora_dropout = 0.1

bias = "none"

task_type = TaskType.SEQ_CLS

Count trainable parameters using:

sum(p.numel() for p in model.parameters() if p.requires_grad)

How many parameters are trainable?

In [14]:
from peft import LoraConfig, TaskType, get_peft_model

config = LoraConfig(
    r = 8,
    lora_alpha = 16,
    target_modules = ["query", "value"],
    lora_dropout = 0.1,
    bias = "none",
    task_type = TaskType.SEQ_CLS

)

model = get_peft_model(model, config)

trainable_params = sum(
    p.numel() for p in model.parameters() if p.requires_grad
)

print(trainable_params)

295681


Q8. Hugging Face Dataset Preparation

Create a Hugging Face Dataset from the first 100 rows of train.csv.

For each row, create:

input_ids with shape [5, 128]

attention_mask with shape [5, 128]

labels as the encoded answer label

For the first dataset item, input_ids has shape:
[5, 128]

How many tokenized choices are stored in input_ids?

In [15]:
from datasets import Dataset

train100_df = train_df.head(100)

dataset = Dataset.from_pandas(train100_df)

In [16]:
def tokenize(data):
  prompt = data["prompt"]

  options = [prompt + "[SEP]" + data[item] for item in label_map.keys()]


  encoding = tokenizer(
      options,
      padding = "max_length",
      max_length = 128,
      truncation = True,
  )

  return {
      "input_ids": encoding["input_ids"],
      "attention_mask": encoding["attention_mask"],
      "labels": data["answer"]
  }

In [17]:
dataset = dataset.map(tokenize)

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

In [18]:
dataset

Dataset({
    features: ['id', 'prompt', 'A', 'B', 'C', 'D', 'E', 'answer', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 100
})

In [19]:
dataset = dataset.remove_columns(
    ["prompt", 'A', 'B', 'C', 'D', 'E', 'answer']
)

In [20]:
len(dataset[0]["input_ids"]), len(dataset[0]["input_ids"][0])

(5, 128)

Q9. Tiny LoRA Fine-Tuning

Fine-tune a LoRA multiple-choice model on the first 32 rows using Hugging Face Trainer.

Use the following settings:

max_length = 64

per_device_train_batch_size = 4

gradient_accumulation_steps = 1

max_steps = 4

What is the final global_step reported by the Trainer?

In [21]:
from transformers import Trainer, TrainingArguments

In [22]:
def tokenize(data):
  prompt = data["prompt"]

  options = [prompt + " [SEP] " + data[item] for item in label_map.keys()]


  encoding = tokenizer(
      options,
      padding = "max_length",
      max_length = 64,
      truncation = True,
  )

  return {
      "input_ids": encoding["input_ids"],
      "attention_mask": encoding["attention_mask"],
      "labels": data["answer"]
  }

In [23]:
train32_df = train_df.head(32)

dataset = Dataset.from_pandas(train32_df)
dataset = dataset.map(tokenize)
dataset = dataset.remove_columns(
    ["prompt", 'A', 'B', 'C', 'D', 'E', 'answer']
)

dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)


training_arguments = TrainingArguments(
    output_dir = "./outputs",
    per_device_train_batch_size = 4,
    gradient_accumulation_steps = 1,
    max_steps = 4,

)

trainer = Trainer(
    model = model,
    args = training_arguments,
    train_dataset = dataset
)

Map:   0%|          | 0/32 [00:00<?, ? examples/s]

In [25]:
import torch
import torchvision
import datasets

print("torch:", torch.__version__)
print("torchvision:", torchvision.__version__)
print("datasets:", datasets.__version__)

torch: 2.11.0+cu128
torchvision: 0.26.0+cu128
datasets: 5.0.0


In [27]:
trainer.train()

print(f"Global step = {trainer.state.global_step}")

[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss


Global step = 4


Q10. Probability Assigned to Option E After Fine-Tuning
Using the fine-tuned LoRA model from Q9, run inference on row index 0 and apply softmax to the logits.

What is the probability assigned to Option E?

Round your answer to 4 decimal places.